# imports

In [32]:
import os
import sys

BASE_DIR = os.path.abspath("..")

sys.path.append(BASE_DIR)
sys.path.append(os.path.join(BASE_DIR, "spark"))

from spark.manager import get_spark

In [33]:
spark = get_spark()

spark

In [34]:
spark.sql("SHOW NAMESPACES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
+---------+



In [35]:
spark.sql("SHOW TABLES IN local.lakehouse").show(truncate=False)


+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|lakehouse|customers  |false      |
|lakehouse|orders     |false      |
|lakehouse|dim_date   |false      |
|lakehouse|products   |false      |
|lakehouse|order_items|false      |
|lakehouse|stores     |false      |
+---------+-----------+-----------+



In [36]:
TABLE = "local.lakehouse.order_items"

# Examples:
# TABLE = "local.lakehouse.order_items"
# TABLE = "local.lakehouse.products"

In [37]:
spark.sql(f"DESCRIBE TABLE {TABLE}").show()

+----------+-------------+-------+
|  col_name|    data_type|comment|
+----------+-------------+-------+
|   item_id|          int|   NULL|
|  order_id|          int|   NULL|
|product_id|          int|   NULL|
|  quantity|          int|   NULL|
|unit_price|decimal(10,2)|   NULL|
|  discount| decimal(5,2)|   NULL|
|line_total|decimal(12,2)|   NULL|
|created_at|    timestamp|   NULL|
+----------+-------------+-------+



In [38]:
spark.sql(f"""
SELECT *
FROM {TABLE}
LIMIT 30
""").show(truncate=False)

+-------+--------+----------+--------+----------+--------+----------+-------------------+
|item_id|order_id|product_id|quantity|unit_price|discount|line_total|created_at         |
+-------+--------+----------+--------+----------+--------+----------+-------------------+
|1      |1       |19        |3       |3046.04   |15.00   |7767.40   |2026-06-02 20:36:58|
|1      |1       |19        |3       |3046.04   |15.00   |7767.40   |2026-06-02 20:36:58|
|1      |1       |19        |3       |3046.04   |15.00   |7767.40   |2026-06-02 20:36:58|
|1      |1       |19        |3       |3046.04   |15.00   |7767.40   |2026-06-02 20:36:58|
|1      |1       |19        |3       |3046.04   |15.00   |7767.40   |2026-06-02 20:36:58|
|1      |1       |19        |3       |3046.04   |15.00   |7767.40   |2026-06-02 20:36:58|
|1      |1       |19        |3       |3046.04   |15.00   |7767.40   |2026-06-02 20:36:58|
|1      |1       |19        |3       |3046.04   |15.00   |7767.40   |2026-06-02 20:36:58|
|1      |1

In [39]:
spark.sql(f"""
SELECT *
FROM {TABLE}.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                               

In [40]:
spark.sql(f"""
SELECT
    snapshot_id,
    committed_at,
    operation
FROM {TABLE}.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|1739502491776378745|2026-07-01 16:08:37.005|replace  |
|5140786520732104232|2026-07-01 16:08:36.578|replace  |
|472718170801614830 |2026-07-01 16:08:00.616|append   |
|7470970373448173629|2026-07-01 16:08:00.333|append   |
|7926443859178995196|2026-07-01 16:07:59.768|append   |
+-------------------+-----------------------+---------+



In [41]:
spark.sql(f"""
SELECT
    file_path,
    file_size_in_bytes,
    record_count
FROM {TABLE}.files
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+------------+
|file_path                                                                                                                                                           |file_size_in_bytes|record_count|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+------------+
|D:/Bootcamp_Project/lakehouse-maintenance-copilot/backend/spark/warehouse/lakehouse/order_items/data/00000-4093-bbec791b-5580-4d39-b057-a9378533ba19-0-00001.parquet|832736            |84606       |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+------------+



In [42]:
spark.sql(f"""
SELECT
    COUNT(*) AS file_count,
    ROUND(AVG(file_size_in_bytes)/1024,2) AS avg_file_kb,
    ROUND(MAX(file_size_in_bytes)/1024,2) AS max_file_kb,
    ROUND(MIN(file_size_in_bytes)/1024,2) AS min_file_kb,
    ROUND(SUM(file_size_in_bytes)/1024/1024,2) AS total_size_mb
FROM {TABLE}.files
""").show()

+----------+-----------+-----------+-----------+-------------+
|file_count|avg_file_kb|max_file_kb|min_file_kb|total_size_mb|
+----------+-----------+-----------+-----------+-------------+
|         1|     813.22|     813.22|     813.22|         0.79|
+----------+-----------+-----------+-----------+-------------+



In [43]:
spark.sql(f"""
SELECT *
FROM {TABLE}.all_manifests
""").show(truncate=False)

+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+-------------------+---------------------+------------+
|content|path                                                                                                                                                 |length|partition_spec_id|added_snapshot_id  |added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|partition_summaries|reference_snapshot_id|key_metadata|
+-------+-----------------------------------------------------------------------------------------------------------------------------------------------------+------+------

In [44]:
spark.sql(f"""
SELECT COUNT(*) AS manifest_count
FROM {TABLE}.all_manifests
""").show()

+--------------+
|manifest_count|
+--------------+
|           106|
+--------------+



In [45]:
spark.sql(f"""
SELECT *
FROM {TABLE}.metadata_log_entries
ORDER BY timestamp DESC
""").show(truncate=False)

+-----------------------+---------------------------------------------------------------------------------------------------------------------------+-------------------+----------------+----------------------+
|timestamp              |file                                                                                                                       |latest_snapshot_id |latest_schema_id|latest_sequence_number|
+-----------------------+---------------------------------------------------------------------------------------------------------------------------+-------------------+----------------+----------------------+
|2026-07-01 16:17:04.85 |D:/Bootcamp_Project/lakehouse-maintenance-copilot/backend/spark/warehouse/lakehouse/order_items/metadata/v354.metadata.json|1739502491776378745|0               |349                   |
|2026-07-01 16:08:37.113|D:/Bootcamp_Project/lakehouse-maintenance-copilot/backend/spark/warehouse/lakehouse/order_items/metadata/v353.metadata.json|17395024917

In [46]:
spark.sql(f"""
SELECT *
FROM {TABLE}.history
ORDER BY made_current_at DESC
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-07-01 16:08:37.005|1739502491776378745|5140786520732104232|true               |
|2026-07-01 16:08:36.578|5140786520732104232|472718170801614830 |true               |
|2026-07-01 16:08:00.616|472718170801614830 |7470970373448173629|true               |
|2026-07-01 16:08:00.333|7470970373448173629|7926443859178995196|true               |
|2026-07-01 16:07:59.768|7926443859178995196|7477254339235527760|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [47]:
spark.sql(f"""
SELECT *
FROM {TABLE}.partitions
""").show(truncate=False)

+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|84606       |1         |832736                       |0                           |0                         |0                           |0                         |2026-07-01 16:08:36.578|5140786520732104232     |
+------------+----------+-----------------------------+----------------------------+--------------------------+---------------------

In [48]:
spark.sql(f"""
SELECT
    snapshot_id,
    manifest_list
FROM {TABLE}.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |manifest_list                                                                                                                                                                |
+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1739502491776378745|D:/Bootcamp_Project/lakehouse-maintenance-copilot/backend/spark/warehouse/lakehouse/order_items/metadata/snap-1739502491776378745-1-4889ea8c-e5c8-418a-afb6-6ab567522b12.avro|
|5140786520732104232|D:/Bootcamp_Project/lakehouse-maintenance-copilot/backend/spark/warehouse/lakehouse/order_items/metadata/snap-5140786520732104232-1-c6766919-c6d6-4252-bc4b-93830fcee65a.avro|
|472718170801614830 

In [49]:
spark.sql(f"""
SELECT COUNT(*)
FROM {TABLE}
""").show()

+--------+
|count(1)|
+--------+
|   84606|
+--------+



In [50]:
spark.sql(f"""
SELECT
    COUNT(*) AS data_files,
    ROUND(AVG(file_size_in_bytes)/1024,2) AS avg_file_kb,
    ROUND(SUM(file_size_in_bytes)/1024/1024,2) AS total_size_mb
FROM {TABLE}.files
""").show()

+----------+-----------+-------------+
|data_files|avg_file_kb|total_size_mb|
+----------+-----------+-------------+
|         1|     813.22|         0.79|
+----------+-----------+-------------+



In [51]:
spark.sql(f"""
SELECT *
FROM {TABLE}
""").explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'UnresolvedRelation [local, lakehouse, order_items], [], false

== Analyzed Logical Plan ==
item_id: int, order_id: int, product_id: int, quantity: int, unit_price: decimal(10,2), discount: decimal(5,2), line_total: decimal(12,2), created_at: timestamp
Project [item_id#1390, order_id#1391, product_id#1392, quantity#1393, unit_price#1394, discount#1395, line_total#1396, created_at#1397]
+- SubqueryAlias local.lakehouse.order_items
   +- RelationV2[item_id#1390, order_id#1391, product_id#1392, quantity#1393, unit_price#1394, discount#1395, line_total#1396, created_at#1397] local.lakehouse.order_items

== Optimized Logical Plan ==
RelationV2[item_id#1390, order_id#1391, product_id#1392, quantity#1393, unit_price#1394, discount#1395, line_total#1396, created_at#1397] local.lakehouse.order_items

== Physical Plan ==
*(1) ColumnarToRow
+- BatchScan local.lakehouse.order_items[item_id#1390, order_id#1391, product_id#1392, quantity#1393, unit_price#139

In [52]:
spark.stop()